<a href="https://colab.research.google.com/github/wasizafar/ai_resume_job_matcher/blob/main/data/data_processing/extract_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install necessary libraries
pip install PyMuPDF opencv-python easyocr

In [16]:
import os
import io
import fitz # PyMuPDF for PDF handling
import cv2 # OpenCV for image processing
import numpy as np
import pandas as pd
from PIL import Image # Pillow for image manipulation
import easyocr # GPU-accelerated OCR library
import concurrent.futures # For parallel processing
from tqdm.auto import tqdm # For progress bar

In [11]:
def preprocess_image_for_ocr(img):
    # Convert to grayscale
    gray_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)

    # Binarization (thresholding) using OTSU
    _, binary_img = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Denoising
    denoised_img = cv2.medianBlur(binary_img, 3)

    return Image.fromarray(denoised_img)

In [12]:
def get_ocr_reader():
    # Initialize EasyOCR reader with GPU support
    # This function is called within each process to ensure proper GPU utilization
    return easyocr.Reader(['en'], gpu=True)

In [13]:
def extract_text_from_pdf_page_easyocr(page, reader):
    # Render page to an image with high DPI for better OCR accuracy
    pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
    img_bytes = pix.tobytes("png")
    pil_img = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    # Preprocess the image
    processed_img = preprocess_image_for_ocr(pil_img)

    # Perform OCR using EasyOCR
    results = reader.readtext(np.array(processed_img))

    # Extract text from EasyOCR results and join them
    page_text = " ".join([res[1] for res in results])
    return page_text

In [14]:
def process_single_pdf_file(task_args):
    category, file, pdf_path = task_args
    # Initialize reader within each process to properly utilize GPU for each worker
    reader = get_ocr_reader()
    ocr_text_pages = []

    try:
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            page_text = extract_text_from_pdf_page_easyocr(page, reader)
            if page_text.strip():
                ocr_text_pages.append(page_text.strip())
        doc.close()
        return [category, file, "\n".join(ocr_text_pages).strip()]
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        return [category, file, ""]

# PDF Text Extraction with GPU Acceleration

In [ ]:
input_folder = '/content/drive/MyDrive/resumes_pdf'

all_pdf_tasks = []

# Collect all PDF file paths to process
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith('.pdf'):
            category = os.path.basename(root)
            pdf_path = os.path.join(root, file)
            all_pdf_tasks.append((category, file, pdf_path))

print(f"Found {len(all_pdf_tasks)} PDF files to process.")

data_rows = []

# Use ProcessPoolExecutor for parallel processing across multiple CPU cores
# Each process will initialize its own EasyOCR reader, leveraging its dedicated GPU resources
with concurrent.futures.ProcessPoolExecutor() as executor:
    # Wrap the executor.map with tqdm for a progress bar
    for result in tqdm(executor.map(process_single_pdf_file, all_pdf_tasks), total=len(all_pdf_tasks), desc="Processing PDFs"):
        data_rows.append(result)

# Create a DataFrame from the extracted data
df_extracted_gpu = pd.DataFrame(data_rows, columns=['Category', 'Filename', 'Plain Text'])

display(df_extracted_gpu.head())

Found 8913 PDF files to process.


Processing PDFs:   0%|          | 0/8913 [00:00<?, ?it/s]

In [ ]:
# Save the extracted dataset to a CSV file
output_csv_path_gpu = '/content/extracted_pdf_text_gpu_accelerated.csv'
df_extracted_gpu.to_csv(output_csv_path_gpu, index=False, encoding='utf-8')
print(f'Successfully saved extracted text to {output_csv_path_gpu}')